# Methane caloric properties

**Evidence: numerical verification.** Evaluate methane vapor at 298.15 K and 100,000 Pa.


Gross and Sadowski (2001), Table 2
([paper](https://doi.org/10.1021/ie0003887)), supplies the methane coefficients
in `gross-2001-methane-ethane/3`. McBride, Zehe and Gordon (2002),
[NASA/TP-2002-211556](https://ntrs.nasa.gov/citations/20020085330), Appendix D,
supplies the NASA9 methane record in `nasa-glenn-2002-hda-species/1`.
The packet preserves the source gas constant, 8.314510 J/(mol K), and
100,000 Pa standard pressure; `ThermochemistryPacket` performs the native
gas-constant conversion. The first methane interval is 200–1000 K.
Zotero parent: `AQDUURW4`; PDF: `SYJLCAXZ`.


## Read the inputs

Use a repository checkout with the environment described in the installation
guide. The wheel alone does not contain these data. All density, association,
equilibrium and parameter-fit iterations below run in the native Engine.


In [1]:
from pathlib import Path
import csv
import pandas as pd
from IPython.display import display
from epcsaft import (
    Parameters,
    Pure,
    ThermochemistryPacket,
    ThermochemistryStatus,
    GAS_CONSTANT_J_PER_MOL_K,
    PropertyObservable,
    PressureActionDirection,
    PropertyActionStatus,
)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

ROOT = next(
    folder
    for folder in (Path.cwd(), *Path.cwd().parents)
    if (folder / "ARCHITECTURE.yaml").is_file()
)
PACKETS = ROOT / "data" / "packets"


## Attach the ideal thermal record

Residual EOS parameters and ideal thermal coefficients supply different physical
information. Select methane from both files in the same component order.


In [2]:
parameters = Parameters.from_json(
    PACKETS / "gross-2001-methane-ethane/3/parameters/parameters.json",
    components=["methane"],
)
thermal = ThermochemistryPacket(
    ROOT / "data/packets/nasa-glenn-2002-hda-species/1/parameters/nasa9-records.json"
).select(["methane"])
model = Pure(parameters, thermochemistry=thermal)


## Compare caloric quantities and an exact action

The Engine returns Ideal, Residual and Total properties. Compare the ideal
values at the ideal standard density with the retained NASA9 reference row:
enthalpy within 0.001 J/mol, Cp and entropy within 10⁻⁶ J/(mol K).

For a dimensionless path variable, the action of enthalpy along a **1 K**
temperature increment has units J/mol. Divide by that increment to obtain
the partial derivative in J/(mol K). At fixed pressure this equals Cp;
require a difference below 10⁻¹⁰ J/(mol K).


In [3]:
state = model.state(298.15, P=100000.0, phase="vapor", derived_properties=True)
enthalpy_actions = model.pressure_actions(
    298.15,
    100000.0,
    phase="vapor",
    observables=[PropertyObservable.TotalEnthalpy],
    directions=[PressureActionDirection(temperature=1)],
)
assert enthalpy_actions.results[0].status == PropertyActionStatus.Available
temperature_increment_K = 1.0
dh_dT = enthalpy_actions.results[0].action / temperature_increment_K
error = dh_dT - state.total_isobaric_heat_capacity
assert abs(error) < 1e-10
reference_state = model.state(
    298.15, rho=100000.0 / (GAS_CONSTANT_J_PER_MOL_K * 298.15)
)
with (
    ROOT / "analyses/2026-hda-two-flash/results/nasa9_reference_check.csv"
).open() as stream:
    reference = next(
        (
            row
            for row in csv.DictReader(stream)
            if row["species"] == "methane" and float(row["T_K"]) == 298.15
        )
    )
assert (
    abs(reference_state.ideal_isobaric_heat_capacity - float(reference["cp_expected"]))
    < 1e-06
)
assert abs(reference_state.ideal_enthalpy - float(reference["h_expected"])) < 0.001
assert abs(reference_state.ideal_entropy - float(reference["s_expected"])) < 1e-06
values = dict(
    h_J_mol=state.total_enthalpy,
    cp_J_mol_K=state.total_isobaric_heat_capacity,
    dh_dT_J_mol_K=dh_dT,
    identity_error_J_mol_K=error,
    speed_m_s=state.speed_of_sound,
    ideal_cp_J_mol_K=reference_state.ideal_isobaric_heat_capacity,
    ideal_h_J_mol=reference_state.ideal_enthalpy,
    ideal_s_J_mol_K=reference_state.ideal_entropy,
)
display(
    pd.DataFrame(
        {
            "Quantity": [
                "Enthalpy / J mol^-1",
                "Cp / J mol^-1 K^-1",
                "Entropy / J mol^-1 K^-1",
            ],
            "Ideal": [
                state.ideal_enthalpy,
                state.ideal_isobaric_heat_capacity,
                state.ideal_entropy,
            ],
            "Residual": [
                state.residual_enthalpy,
                state.residual_isobaric_heat_capacity,
                state.residual_entropy,
            ],
            "Total": [
                state.total_enthalpy,
                state.total_isobaric_heat_capacity,
                state.total_entropy,
            ],
        }
    )
)
without_thermal = Pure(parameters).state(298.15, P=100000, phase="vapor")
assert without_thermal.total_enthalpy is None
assert without_thermal.thermochemistry_status == ThermochemistryStatus.MissingRecord
values["nasa_reference_errors"] = {
    "h_J_mol": reference_state.ideal_enthalpy - float(reference["h_expected"]),
    "cp_J_mol_K": reference_state.ideal_isobaric_heat_capacity
    - float(reference["cp_expected"]),
    "s_J_mol_K": reference_state.ideal_entropy - float(reference["s_expected"]),
}
display(pd.DataFrame([values["nasa_reference_errors"]]))


,Quantity,Ideal,Residual,Total
0,Enthalpy / J mol^-1,-74599.999986,-16.276263,-74616.276249
1,Cp / J mol^-1 K^-1,35.691326,0.093520,35.784846
2,Entropy / J mol^-1 K^-1,186.355438,-0.023333,186.332106


,h_J_mol,cp_J_mol_K,s_J_mol_K
0,-2.037268e-10,-3.077787e-10,-4.536105e-11


In [4]:
display(pd.DataFrame([{
    "Normalized dh/dT / J mol^-1 K^-1": dh_dT,
    "dh/dT minus Cp / J mol^-1 K^-1": error,
    "Sound speed / m s^-1": values["speed_m_s"],
    "NASA reference ideal Cp / J mol^-1 K^-1": values["ideal_cp_J_mol_K"],
    "NASA reference ideal h / J mol^-1": values["ideal_h_J_mol"],
    "NASA reference ideal s / J mol^-1 K^-1": values["ideal_s_J_mol_K"],
}]))


,Normalized dh/dT / J mol^-1 K^-1,dh/dT minus Cp / J mol^-1 K^-1,Sound speed / m s^-1,NASA reference ideal Cp / J mol^-1 K^-1,NASA reference ideal h / J mol^-1,NASA reference ideal s / J mol^-1 K^-1
0,35.784846,-7.105427e-15,448.44759,35.691326,-74599.999986,186.371072


## Meaning and limits


The reference comparison checks transcription and evaluation of NASA9 inputs;
the derivative identity checks numerical consistency. Neither is a comparison
with calorimetric measurements. Total enthalpy includes the residual correction
at the real gas density, whereas ideal entropy depends on the declared density
and reference pressure. Without a thermal record, total calorics are unavailable.
